## synthetic dataset generation code

In [17]:
import numpy as np
import pandas as pd

def generate_archery_dataset(num_shots=200):
    np.random.seed(42)

    # -------------------------------------------------------------
    # Hyperparameters & System Coefficients
    # -------------------------------------------------------------
    alpha = 0.025  # Rate of fatigue accumulation per shot unit
    beta = 0.005   # Impact of increasing heart rate on physical stress
    gamma = 0.0004 # Efficiency of recovery time on draining fatigue
    
    # -------------------------------------------------------------
    # Skill Priors & Static Traits (Layer 1 - Initialization)
    # -------------------------------------------------------------
    experience = np.random.uniform(2, 15)       # Experience in years
    skill_consistency = np.random.uniform(0.6, 0.95) # Baseline skill level (0-1)

    # Latent state tracking initialization at t=0
    fatigue_level = 0.1          # Scale 0-1
    baseline_heart_rate = 70.0  
    time_since_start = 0.0       # Cumulative clock time in seconds
    previous_performance_state = 0.0 # Starts neutral (Scale -5 to +5)
    previous_shot_variability = 0.1
    
    # Historical queue to track the last 5 shots for rolling performance metrics
    shot_history = []
    dataset = []

    for shot in range(1, num_shots + 1):
        # 4) Context Features: Temporal Progression
        shot_number = shot
        round_number = int((shot - 1) // 6) + 1  # 6 shots per standard end/round
        
        # Simulating rest period and time accumulation
        recovery_time_before_shot = np.random.uniform(30, 90) # seconds
        time_since_start += recovery_time_before_shot + 15.0 # Adds 15s setup window per shot

        # -------------------------------------------------------------
        # 1) Core Latent State Layer
        # -------------------------------------------------------------
        
        # A. Mental Cognitive State
        # Pressure increases as tournament progresses, modulated by experience dampening
        base_pressure = (shot_number / num_shots) * 0.5 + np.random.uniform(0.1, 0.3)
        pressure_level = np.clip(base_pressure / (1.0 + (experience * 0.05)), 0.0, 1.0)
        
        # Performance drop under pressure scales negatively with skill level
        performance_drop_under_pressure = max(0.0, (pressure_level - 0.5) * (1.0 - skill_consistency))
        
        # B. Physical State Logic (Temporal Accumulation with Dynamic Memory)
        shot_load = shot ** 0.15
        
        # Heart rate response drifts up organically based on baseline fatigue and mental pressure
        heart_rate_change = np.random.uniform(5, 20) + (fatigue_level * 40) + (pressure_level * 15)
        heart_rate_during_shot = baseline_heart_rate + heart_rate_change
        
        # State Transitions: Past State + Exertion Gain + Stress Gain - Recovery Drain
        fatigue_gain = (alpha * shot_load) + (beta * heart_rate_change)
        recovery_drain = fatigue_level * (gamma * recovery_time_before_shot)
        fatigue_level = np.clip(fatigue_level + fatigue_gain - recovery_drain, 0.0, 1.0)
        
        # Comprehensive Fatigue State Function
        fatigue_state = fatigue_level + (shot_number * 0.001) + (time_since_start * 0.0001) + (heart_rate_change * 0.002) - (recovery_time_before_shot * 0.0005)
        recovery_response = recovery_drain / (fatigue_gain + 1e-6)

        # C. Combined State Coupled Interactions
        pressure_response = pressure_level * 0.4 + (heart_rate_change / 100.0) * 0.4 + performance_drop_under_pressure * 0.2
        
        # -------------------------------------------------------------
        # 3) Environment Variables Layer
        # -------------------------------------------------------------
        wind_speed = np.random.uniform(0, 25)       # km/hr
        wind_direction = np.random.uniform(0, 360)   # degrees
        wind_gust_variability = np.random.uniform(0, 8) # km/hr
        
        # Magnitude mapping for external disturbance
        environmental_disturbance_index = (wind_speed * np.abs(np.sin(np.radians(wind_direction)))) + wind_gust_variability
        
        # Thermal context environment
        temperature = np.random.uniform(18, 34) # Celsius
        humidity = np.random.uniform(40, 85)    # Percentage
        thermal_stress = (temperature * 0.5 + humidity * 0.2 + (time_since_start / 3600.0) * 0.3)

        # -------------------------------------------------------------
        # 1-B) Motor Stability State & 2) Physical Mechanism Features
        # -------------------------------------------------------------
        base_tremor = np.random.normal(1.0, 0.05)
        
        # Non-Linear Compensation Threshold Boundary Check
        if fatigue_level > 0.75:
            # Beyond 0.75: Complete structural breakdown
            compensation_breakdown_factor = np.exp((fatigue_level - 0.75) * 5)
            tremor_level = base_tremor * compensation_breakdown_factor
            
            # Mechanical consequences erode exponentially
            release_angle_error = np.random.normal(0.05, 0.01) * (1.0 + fatigue_level * 5)
            bow_alignment_variability = np.random.normal(0.1, 0.02) * (1.0 + fatigue_level * 4)
            anchor_variation = np.random.normal(0.5, 0.1) * (1.0 + fatigue_level * 3)
            force_consistency = np.random.normal(50.0, 5.0) / (1.0 + fatigue_level * 0.5)
        else:
            # Before 0.75: Active biological & cognitive stabilization
            tremor_level = base_tremor
            release_angle_error = np.random.normal(0.05, 0.01) * 1.1
            bow_alignment_variability = np.random.normal(0.1, 0.02) * 1.05
            anchor_variation = np.random.normal(0.5, 0.05)
            force_consistency = np.random.normal(50.0, 1.5)

        # Additional latent target properties
        stability = 1.0 - np.clip((tremor_level + bow_alignment_variability + anchor_variation) / 10.0, 0.0, 1.0)
        motor_stability_state = tremor_level * 0.4 + bow_alignment_variability * 0.3 + anchor_variation * 0.2 + release_angle_error * 0.1
        stability_state = np.sin(shot_number) * stability  # Models sequenced oscillation profiles

        # -------------------------------------------------------------
        # 4 & 5) Output Target Mapping & Uncertainty Mechanics
        # -------------------------------------------------------------
        shot_noise_factor = np.random.uniform(0.0, 1.0)
        unexplained_variability = shot_noise_factor * 0.3 # Random noise buffer
        
        # Final Execution Equation linking Hidden States, Mechanics, Environment and Target
        arrow_distance_from_center = (
            (tremor_level * 2.5) + 
            (release_angle_error * 12.0) + 
            (bow_alignment_variability * 8.0) +
            (anchor_variation * 1.5) +
            (environmental_disturbance_index * 0.15) +
            (performance_drop_under_pressure * 3.0) -
            (force_consistency * 0.02) + 
            np.random.normal(0, 0.2) + 
            unexplained_variability
        )
        
        # Handle logical ceiling constraints for physical outputs
        arrow_distance_from_center = max(0.0, arrow_distance_from_center)
        
        # Calculate trailing history indicators (Rolling metrics based on last 5 shots)
        shot_history.append(arrow_distance_from_center)
        if len(shot_history) > 5:
            shot_history.pop(0)
            
        rolling_mean = np.mean(shot_history)
        rolling_variance = np.var(shot_history) if len(shot_history) > 1 else 0.0
        rolling_trend = (shot_history[-1] - shot_history[0]) if len(shot_history) > 1 else 0.0
        rolling_performance_state = {"mean": rolling_mean, "variance": rolling_variance, "trend": rolling_trend}
        
        # Track historical momentum state metrics for the NEXT loop sequence
        if shot > 1:
            momentum_feature = rolling_trend
            previous_performance_state = np.clip(rolling_trend * -1, -5.0, 5.0) # Mapping bad trends to negative states
            previous_shot_variability = np.abs(arrow_distance_from_center - rolling_mean)
        else:
            momentum_feature = 0.0
            
        # Append finalized structure row to the master collection matrix
        dataset.append({
            # 4) Context / Telemetry Group
            "shot_number": shot_number,
            "round_number": round_number,
            "time_since_start_seconds": time_since_start,
            
            # 1) Latent Physical & Skill State Metrics
            "experience_years": experience,
            "skill_consistency": skill_consistency,
            "fatigue_level": fatigue_level,
            "fatigue_state": fatigue_state,
            "recovery_time_before_shot": recovery_time_before_shot,
            "recovery_response": recovery_response,
            "heart_rate_change_bpm": heart_rate_change,
            "heart_rate_during_shot": heart_rate_during_shot,
            
            # 1) Cognitive States
            "pressure_level": pressure_level,
            "performance_drop_under_pressure": performance_drop_under_pressure,
            "pressure_response": pressure_response,
            "momentum_feature": momentum_feature,
            "previous_performance_state": previous_performance_state,
            
            # 1 & 2) Motor Control / Mechanical Features
            "tremor_level": tremor_level,
            "stability_score": stability,
            "motor_stability_state": motor_stability_state,
            "stability_state_sequence": stability_state,
            "release_angle_error_deg": release_angle_error,
            "bow_alignment_variability_deg": bow_alignment_variability,
            "anchor_variation_mm": anchor_variation,
            "force_consistency_newtons": force_consistency,
            
            # 3) Environment Metrics
            "wind_speed_kmhr": wind_speed,
            "wind_direction_deg": wind_direction,
            "wind_gust_variability_kmhr": wind_gust_variability,
            "environmental_disturbance_index": environmental_disturbance_index,
            "thermal_stress_index": thermal_stress,
            
            # 4 & 5) Noise & Outputs Target Group
            "shot_noise_factor": shot_noise_factor,
            "unexplained_variability": unexplained_variability,
            "rolling_performance_mean": rolling_performance_state["mean"],
            "rolling_performance_variance": rolling_performance_state["variance"],
            "rolling_performance_trend": rolling_performance_state["trend"],
            "previous_shot_variability": previous_shot_variability,
            "arrow_distance_from_center_cm": arrow_distance_from_center
        })

    return pd.DataFrame(dataset)

# Execute verification buildno
df_archery = generate_archery_dataset(500)
print(f"Generated Dataset Shape: {df_archery.shape}")  

Generated Dataset Shape: (500, 36)


# Exploratory data analysis

## 1. original feature set

In [18]:
df_archery.head()

,shot_number,round_number,time_since_start_seconds,experience_years,skill_consistency,fatigue_level,fatigue_state,recovery_time_before_shot,recovery_response,heart_rate_change_bpm,...,wind_gust_variability_kmhr,environmental_disturbance_index,thermal_stress_index,shot_noise_factor,unexplained_variability,rolling_performance_mean,rolling_performance_variance,rolling_performance_trend,previous_shot_variability,arrow_distance_from_center_cm
0,1,1,88.919637,6.869022,0.93275,0.191067,0.191609,73.919637,0.031447,13.804810,...,6.929409,8.321281,28.188983,0.431945,0.129584,4.638010,0.000000,0.000000,0.100000,4.638010
1,2,1,151.393385,6.869022,0.93275,0.301380,0.329262,47.473748,0.031843,17.240258,...,3.648560,9.085555,25.091088,0.097672,0.029302,5.170745,0.283807,1.065470,0.532735,5.703480
2,3,1,237.447367,6.869022,0.93275,0.427386,0.460641,71.053982,0.063651,21.018662,...,7.274563,9.928614,25.052728,0.088493,0.026548,5.059504,0.213954,0.199010,0.222483,4.837020
3,4,1,294.206338,6.869022,0.93275,0.592213,0.661230,41.758972,0.041513,28.237572,...,6.629900,16.259542,22.406954,0.074045,0.022213,5.388657,0.485491,1.738107,0.987460,6.376118
4,5,1,360.714282,6.869022,0.93275,0.827169,0.928619,51.507944,0.049367,43.066184,...,0.508467,14.120900,22.444568,0.760785,0.228236,7.463073,17.601192,11.122724,8.297662,15.760735


the dataset features are divided into 5 mechanisms: 1/ Physiological features
2/ Environmental features
3/ Biomechanical features
4/ Contextual features
5/ Target variable

each mechanism influence other mechanism, feature influence features, and resultant performance emerges in the end

feature names

In [19]:
print("feature names:")
df_archery.columns.tolist()

feature names:


['shot_number',
 'round_number',
 'time_since_start_seconds',
 'experience_years',
 'skill_consistency',
 'fatigue_level',
 'fatigue_state',
 'recovery_time_before_shot',
 'recovery_response',
 'heart_rate_change_bpm',
 'heart_rate_during_shot',
 'pressure_level',
 'performance_drop_under_pressure',
 'pressure_response',
 'momentum_feature',
 'previous_performance_state',
 'tremor_level',
 'stability_score',
 'motor_stability_state',
 'stability_state_sequence',
 'release_angle_error_deg',
 'bow_alignment_variability_deg',
 'anchor_variation_mm',
 'force_consistency_newtons',
 'wind_speed_kmhr',
 'wind_direction_deg',
 'wind_gust_variability_kmhr',
 'environmental_disturbance_index',
 'thermal_stress_index',
 'shot_noise_factor',
 'unexplained_variability',
 'rolling_performance_mean',
 'rolling_performance_variance',
 'rolling_performance_trend',
 'previous_shot_variability',
 'arrow_distance_from_center_cm']

feature datatype

In [20]:
print("datatype")
df_archery.dtypes

datatype


shot_number                          int64
round_number                         int64
time_since_start_seconds           float64
experience_years                   float64
skill_consistency                  float64
fatigue_level                      float64
fatigue_state                      float64
recovery_time_before_shot          float64
recovery_response                  float64
heart_rate_change_bpm              float64
heart_rate_during_shot             float64
pressure_level                     float64
performance_drop_under_pressure    float64
pressure_response                  float64
momentum_feature                   float64
previous_performance_state         float64
tremor_level                       float64
stability_score                    float64
motor_stability_state              float64
stability_state_sequence           float64
release_angle_error_deg            float64
bow_alignment_variability_deg      float64
anchor_variation_mm                float64
force_consi

dataset size

In [21]:
print("number of rows")
df_archery.shape[0]

number of rows


500

In [22]:
print("number of columns:")
df_archery.shape[1]

number of columns:


36

## 2. feature types

### Feature Categorization by Data-Generating Mechanism

| Mechanism | Features | Purpose |
|-----------|----------|---------|
| **1. Context / Temporal Mechanism** | `shot_number`, `round_number`, `time_since_start_seconds`, `previous_performance_state`, `momentum_feature`, `rolling_performance_mean`, `rolling_performance_variance`, `rolling_performance_trend`, `previous_shot_variability` | Represents where the archer is in the competition and how previous shots influence the current shot. |
| **2. Physiological & Psychological Mechanism** | `experience_years`, `skill_consistency`, `fatigue_level`, `fatigue_state`, `recovery_time_before_shot`, `recovery_response`, `heart_rate_change_bpm`, `heart_rate_during_shot`, `pressure_level`, `performance_drop_under_pressure`, `pressure_response` | Represents the internal human state including fatigue, recovery, pressure, and experience. |
| **3. Biomechanical / Motor Control Mechanism** | `tremor_level`, `stability_score`, `motor_stability_state`, `stability_state_sequence`, `release_angle_error_deg`, `bow_alignment_variability_deg`, `anchor_variation_mm`, `force_consistency_newtons` | Represents the physical execution of the shot and motor control. |
| **4. Environmental Mechanism** | `wind_speed_kmhr`, `wind_direction_deg`, `wind_gust_variability_kmhr`, `environmental_disturbance_index`, `thermal_stress_index` | Represents external environmental conditions affecting shot performance. |
| **5. Stochastic / Residual Mechanism** | `shot_noise_factor`, `unexplained_variability` | Represents randomness and latent factors not explicitly modeled. |
| **Target Variable** | `arrow_distance_from_center_cm` | Final outcome generated by the interaction of all five mechanisms. |

### Data-Generating Process

```text
Context & Time
        ↓
Physiological / Psychological State
        ↓
Biomechanical Execution
        ↓
Environmental Disturbances
        ↓
Random / Latent Variability
        ↓
Arrow Distance From Center
```

## 3. feature distribution

### analyze numerical features

In [23]:
numerical_features = df_archery.select_dtypes(include=['int64', 'float64']).columns

In [24]:
print("number of numerical features:", len(numerical_features))
print(numerical_features.tolist())

number of numerical features: 36
['shot_number', 'round_number', 'time_since_start_seconds', 'experience_years', 'skill_consistency', 'fatigue_level', 'fatigue_state', 'recovery_time_before_shot', 'recovery_response', 'heart_rate_change_bpm', 'heart_rate_during_shot', 'pressure_level', 'performance_drop_under_pressure', 'pressure_response', 'momentum_feature', 'previous_performance_state', 'tremor_level', 'stability_score', 'motor_stability_state', 'stability_state_sequence', 'release_angle_error_deg', 'bow_alignment_variability_deg', 'anchor_variation_mm', 'force_consistency_newtons', 'wind_speed_kmhr', 'wind_direction_deg', 'wind_gust_variability_kmhr', 'environmental_disturbance_index', 'thermal_stress_index', 'shot_noise_factor', 'unexplained_variability', 'rolling_performance_mean', 'rolling_performance_variance', 'rolling_performance_trend', 'previous_shot_variability', 'arrow_distance_from_center_cm']


summary statistics for dataset

In [25]:
summary_stats = df_archery[numerical_features].describe().T

summary_stats["Median"] = df_archery[numerical_features].median()
summary_stats["skewness"] = df_archery[numerical_features].skew()

summary_stats = summary_stats[["count", "mean", "std", "25%", "Median", "75%", "max", "skewness"]]

summary_stats = summary_stats.round(3)
summary_stats

,count,mean,std,25%,Median,75%,max,skewness
shot_number,500.0,250.500,144.482,125.750,250.500,375.250,500.000,0.000
round_number,500.0,42.168,24.081,21.000,42.000,63.000,84.000,0.000
time_since_start_seconds,500.0,19194.257,11040.773,9713.832,19196.593,28741.839,38048.157,-0.009
experience_years,500.0,6.869,0.000,6.869,6.869,6.869,6.869,0.000
skill_consistency,500.0,0.933,0.000,0.933,0.933,0.933,0.933,0.000
fatigue_level,500.0,0.995,0.058,1.000,1.000,1.000,1.000,-11.664
fatigue_state,500.0,3.248,1.264,2.183,3.248,4.344,5.392,-0.049
recovery_time_before_shot,500.0,61.096,17.048,45.808,62.719,76.025,89.802,-0.072
recovery_response,500.0,0.071,0.021,0.054,0.072,0.090,0.120,0.036
heart_rate_change_bpm,500.0,57.212,5.675,53.806,57.726,60.887,66.862,-2.325


## 3.1 Dataset overview
### summary of statistical interpretation around mechanisms

since this dataset is organized around mechanisms, here is the pattern interpretation by mechanisms instead of ever single variable individually

### 3.1.1 temporal and context features

**features:** shot_number, round_number, time_since_start_seconds, momentum_feature, previous_performance_state, rolling_performance_mean, rolling_performance_variance, rolling_performance_trend, previous_shot_variability

**interpretation:**
* **shot number**, **round number** and **time since start seconds** have **skewness values close to zero**, indicating that observations are evenly distributed throughout the competition.
* the rolling performance metrics capture the athlete's recent performance history.
  **rolling performance mean** remains relatively stable. **rolling performance variance** shows a **strong positive skew 8.706**, indicating periods of high inconsistency occured infrequently.
* **momentum feature**  and **rolling performance trend** shows **moderate positive skew 1.012**, suggesting that strong positive performance streaks are less. stable performance is present more often.
* **previous shot variability** is also **positively skewed 2.352**, shows that large shot fluctuations are rare.

**observation:** the temporal features suggest that the simulated competition progresses uniformly while preserving occasional periods of unstable performance, reflecting realistic competitive dynamics.

### 3.1.2 physiological and psychological features

**features:** experience_years, skill_consistency, fatigue_level, fatigue_state, recovery_time_before_shot, heart_rate_change_bpm, heart_rate_during_shot, pressure_level, performance_drop_under_pressure, pressure_response

**interpretaion:**
* **experience years** and **skill consistency** remain **constant** throughout the dataset, indicating a single athlete.
* **fatigue level** shows an **extremely negative skew of -11.664**, most observations concentrated near the maximum fatigue, this skewness is well defined as fatigue goes up with tournament progression.
* **fatigue state**, **recovery time before shot**, and **pressure level** are **symmetric** which shows balanced variation in internal physiological states.
* **heart rate variables** are **highly negative skewed -2.325**, showing heart rates dominate most observation. it also shows the athlete cardio levels throughout the tournament.
* **performance drop under pressure** **large positive skew 5.699** indicate that severe performance only occurs under high pressure situations.
* **pressure response** is **slighly left skewed**, indicates stable pshycological pressure response.

**observation:** the physiological features reflect progressive fatigue and increased cardiovascular demand, while realistic variability was preserved in pressure induced performance degradation.

### 3.1.3 biomechanical and motor control stability

**features:** tremor_level, stability_score, motor_stability_state, stability_state_sequence, release_angle_error_deg, bow_alignment_variability_deg, anchor_variation_mm, force_consistency_newtons

**intrpretation:** 
* **tremor level** highly **negative skewed -5.182** and **motor stability state** is also **highly negatively skewed -4.752**, both indicate that athelete generally maintains stable motor control with few severe instability events.
* **stability score** is **positively skewed 2.916** high stability score occurs less frequently.
* **Release angle error**, **bow alignment variability**, and **anchor variation** are **all approximately symmetric**, implying that shot execution errors remain centered around expected operating values.
* **force consistency newtons** shows **mild positive skew 0.621**, indicating occasional deviations in draw force consistency. without repeated varibaility.

**observations:** the biomechanical variable indicate consistent shooting mechanisms while allowing occasional deterioration in motor performance, while some adverse comditions applied.

### 3.1.4 environmental features

**features:** wind_speed_kmhr, wind_direction_deg, wind_gust_variability_kmhr, environmental_disturbance_index, thermal_stress_index

**interpretation:** 
* **Wind speed**, **wind direction**, **gust variability**, and **thermal stress** all exhibit **skewness values close to zero**, indicating balanced environmental sampling across the simulation.
* **environmental_disturbance_index** has **moderate positive skew (0.609)**, reflecting that severe environmental disturbances occur less frequently than mild conditions.

**obseravtions:**
environmental variables provide realistic variation, and avpoid excessive bias towards ideal or extreme conditions.

### randomness variability

**features:** shot noise factor, unexplained variability

**interpretation:** 
* variable display very **small positive skew 0.13**, indicate that stochastic effects are distribiuted uniformly throughout the dataset.
* These variables intentionally introduce uncertainty that cannot be fully explained by the observable physiological, biomechanical, or environmental features.

**observation:** these features provide realistic variability, preventing the datset from becoming perfectly deteministic.

### 3.1.5 target variable

**feature:** arrow distance from center cm 

**interpretation:** 
* the **target variable** has a **mean of approximately 20.35** with **std of 2.16 cm**, indicate moderate dispersion in shot accuracy.
* the distribution is **negatively skewed -2.458**, suggesting that most simulated shots are clustered toward similar accuracy levels, while relatively few observations show larger deviations from center.
  

## overall observation
the descriptive statistics indicate that the synthetic datset successfully captures multiple mechanisms influencing archery performance. most variablee represent realistic tendencies and moderate variability. meaning somewhere close to mirroring real world dataset.

Some intentionally designed features such as, fatigue accumulation, pressure induced performance degradation, rolling performance variance and residual noise introduce asymmetry and rare events. 

**These characterstics produce a dataset** that **resembles human performance dynamics** while remaining suitable **for evaluating the robustness of machine learning models**.

## 3.2 Missing value analysis

In [27]:
missing_value = df_archery.isnull().sum()
missing_value

shot_number                        0
round_number                       0
time_since_start_seconds           0
experience_years                   0
skill_consistency                  0
fatigue_level                      0
fatigue_state                      0
recovery_time_before_shot          0
recovery_response                  0
heart_rate_change_bpm              0
heart_rate_during_shot             0
pressure_level                     0
performance_drop_under_pressure    0
pressure_response                  0
momentum_feature                   0
previous_performance_state         0
tremor_level                       0
stability_score                    0
motor_stability_state              0
stability_state_sequence           0
release_angle_error_deg            0
bow_alignment_variability_deg      0
anchor_variation_mm                0
force_consistency_newtons          0
wind_speed_kmhr                    0
wind_direction_deg                 0
wind_gust_variability_kmhr         0
e

the dataset has no null value. since the data were generated through a controlled simulation pipeline, every observation is complete and no imputation was required.

### 3.3 duplicate values

In [32]:
duplicates = df_archery.duplicated().sum()
print(f"duplicate rows:", duplicates)

duplicate rows: 0


no duplicate observations were found, indicating that every simulated shot represents a unique event